# RUL Predictions

Unlike the notebooks in the "learning_notebooks" folder - this file is supposed to be a polished final approach to the RUL prediction problem, whilst the other notebook act as a small documentation of my learning process.

For this notebook, I decided to use FD004, meaning the dataset that has:
1. 6 unknown operating conditions for the turbines, which we will identify by clustering the data into 6 clusters, using k-means-clustering.
2. 2 fault types instead of just one.

### Imports

In [29]:
import pandas as pd
import matplotlib as plt
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans 

### Loading the training data and RUL

In [30]:
sensors = [f"sensor_{i}" for i in range(1,22)]

columns = (["engine_id", "cycle", "setting_1", "setting_2", "setting_3"] + sensors)
train_data = pd.read_csv("./CMAPSSData/train_FD004.txt", sep=r"\s+", header = None, names=columns)

# The training data always records reading for an engine till failure. 
# Therefore we can calc the Lifetime for each engine AND the RUL by using the max:

lifetime = train_data.groupby("engine_id")["cycle"].transform("max")
train_data["RUL"] = lifetime - train_data["cycle"]

### Splitting up the training data into the 6 operating conditions
We know from the readme.txt that there is 6 operating conditions. Since for each operating conditions sensors will have different standard / reference conditions, we need to split the training set into these 6 groups

In [31]:
# We need to scale the settings data since we will use k-means clustering for unsupervised learning,
# which is not good at handling numbers in different scales

settings_scaler = StandardScaler()
scaled_settings = settings_scaler.fit_transform(train_data[[f"setting_{i}" for i in range(1,4)]])

k_means = KMeans(n_clusters=6, n_init=20, random_state=8)

train_data["regime"] = k_means.fit_predict(scaled_settings)
mean = train_data.groupby("regime")[sensors].mean()
std = train_data.groupby("regime")[sensors].std()

## Filtering out any sensors that have almost 0 variance wihtin their regime
useless_sensors = std.columns[std.mean(axis=0) <= 10e-5].to_list()



### Calculating reference conditions (healthy conditions) for each sensor in each regime
For this I assume that the first 20 cycles of an engines lifetime are healthy and therefore show reference conditions that I can compare later sensor readings with. 

In [32]:
early_training_data = train_data[train_data["cycle"] <= 20]
reference_mean = early_training_data.groupby("regime")[sensors].mean()
reference_std = early_training_data.groupby("regime")[sensors].std()

